# 02 Regime Maps

Purpose:
- Label drift/vol/autocorrelation regimes
- Evaluate conditional return and Sharpe by regime
- Visualize regime heatmaps and drift-vol scatter


In [ ]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Ensure all project-relative IO uses repository root
os.chdir(ROOT)

from src.utils.io import load_config
from src.experiments.run_regimes import run as run_regimes


In [ ]:
CONFIG_PATH = ROOT / "configs" / "default.yaml"
cfg = load_config(CONFIG_PATH)
labeled_by_pair = run_regimes(str(CONFIG_PATH))
print("Regimes generated for:", list(labeled_by_pair.keys()))


In [ ]:
def plot_heatmap(df: pd.DataFrame, title: str, cmap: str = "RdYlBu_r"):
    vals = df.values.astype(float)
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(vals, cmap=cmap, aspect="auto")
    ax.set_xticks(range(df.shape[1]))
    ax.set_yticks(range(df.shape[0]))
    ax.set_xticklabels(df.columns)
    ax.set_yticklabels(df.index)
    ax.set_title(title)
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            if np.isfinite(vals[i, j]):
                ax.text(j, i, f"{vals[i, j]:.4f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


In [ ]:
for name, df in labeled_by_pair.items():
    mean_map = df.pivot_table(index="mu_bin", columns="vol_bin", values="net_ret", aggfunc="mean", observed=False)

    sharpe_map = df.groupby(["mu_bin", "vol_bin"], observed=False)["net_ret"].apply(
        lambda s: (s.mean() / s.std()) * np.sqrt(252) if s.std() > 0 else np.nan
    ).unstack()

    plot_heatmap(mean_map, f"{name}: mean strategy return by regime")
    plot_heatmap(sharpe_map, f"{name}: Sharpe by regime", cmap="coolwarm")


In [ ]:
for name, df in labeled_by_pair.items():
    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(df["drift"], df["vol"], c=df["net_ret"], cmap="coolwarm", alpha=0.75)
    ax.set_title(f"{name}: drift-vol map colored by net return")
    ax.set_xlabel("Drift")
    ax.set_ylabel("Vol")
    ax.grid(alpha=0.2)
    fig.colorbar(sc, ax=ax, label="net_ret")
    plt.show()


In [ ]:
for name, df in labeled_by_pair.items():
    needed = ["regime_2d", "regime_3d", "mu_bin", "vol_bin", "net_ret"]
    assert set(needed).issubset(df.columns), f"{name}: missing regime columns"
    assert df["regime_2d"].notna().all(), f"{name}: regime_2d has missing labels"
    assert df["mu_bin"].notna().all(), f"{name}: mu_bin has missing labels"
    assert df["vol_bin"].notna().all(), f"{name}: vol_bin has missing labels"

print("Regime checks passed.")
